# EfficientNetV2-M + Metadata Hybrid Model (LARGER VERSION)

Upgrade from EfficientNetV2-S to EfficientNetV2-M:
- Parameters: 21.5M → 54M (2.5× larger)
- ImageNet accuracy: 84.3% → 85.1% (+0.8%)
- More capacity = learn more complex patterns

Expected improvement: +0.005-0.015 AUC → 0.942-0.952 LB
Target: Compete for 2nd place, possibly 1st!

**Risk**: Might overfit (more parameters)
**Mitigation**: Same regularization, careful monitoring

In [9]:
import pandas as pd
import numpy as np
import h5py
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_m  # ← MEDIUM instead of SMALL
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
import warnings
import pickle
from datetime import datetime
import json
import seaborn as sns
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():    
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}\n")

Using device: cuda
GPU Memory: 47.7 GB
GPU Name: NVIDIA L40S



## 1. Metadata Preprocessing

In [10]:
data_dir = Path('data')
train_meta = pd.read_csv(data_dir / 'new-train-metadata.csv', low_memory=False)
test_meta = pd.read_csv(data_dir / 'students-test-metadata.csv', low_memory=False)

print("Metadata loaded:")
print(f"  Train: {len(train_meta):,} samples")
print(f"  Test: {len(test_meta):,} samples\n")

# Define features
NUMERICAL_FEATURES = [
    'tbp_lv_H', 'tbp_lv_areaMM2', 'tbp_lv_minorAxisMM',
    'tbp_lv_perimeterMM', 'tbp_lv_deltaB', 'tbp_lv_Hext',
    'clin_size_long_diam_mm', 'tbp_lv_radial_color_std_max',
    'tbp_lv_B', 'tbp_lv_color_std_mean', 'tbp_lv_Aext',
    'tbp_lv_stdLExt', 'tbp_lv_norm_color', 'tbp_lv_A',
    'age_approx'
]

CATEGORICAL_FEATURES = [
    'sex', 'anatom_site_general', 'tbp_tile_type', 'tbp_lv_location_simple'
]

Metadata loaded:
  Train: 400,959 samples
  Test: 100 samples



In [11]:
def preprocess_metadata(df, is_train=True, scaler=None, encoders=None):
    df = df.copy()
    
    for col in NUMERICAL_FEATURES:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median() if is_train else 0)
    
    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            df[col] = df[col].fillna('missing')
    
    if is_train:
        scaler = StandardScaler()
        df[NUMERICAL_FEATURES] = scaler.fit_transform(df[NUMERICAL_FEATURES])
    else:
        df[NUMERICAL_FEATURES] = scaler.transform(df[NUMERICAL_FEATURES])
    
    if is_train:
        encoders = {}
        encoded_dfs = []
        for col in CATEGORICAL_FEATURES:
            encoded = pd.get_dummies(df[col], prefix=col, dtype=float)
            encoders[col] = encoded.columns.tolist()
            encoded_dfs.append(encoded)
        result_df = pd.concat([df[NUMERICAL_FEATURES]] + encoded_dfs, axis=1)
    else:
        encoded_dfs = []
        for col in CATEGORICAL_FEATURES:
            encoded = pd.get_dummies(df[col], prefix=col, dtype=float)
            for train_col in encoders[col]:
                if train_col not in encoded.columns:
                    encoded[train_col] = 0
            encoded = encoded[encoders[col]]
            encoded_dfs.append(encoded)
        result_df = pd.concat([df[NUMERICAL_FEATURES]] + encoded_dfs, axis=1)
    
    return result_df, scaler, encoders

train_meta_processed, scaler, encoders = preprocess_metadata(train_meta, is_train=True)
test_meta_processed, _, _ = preprocess_metadata(test_meta, is_train=False, scaler=scaler, encoders=encoders)

train_meta_processed['isic_id'] = train_meta['isic_id'].values
train_meta_processed['target'] = train_meta['target'].values
test_meta_processed['isic_id'] = test_meta['isic_id'].values

metadata_dim = len(train_meta_processed.columns) - 2
print(f"Metadata dimension: {metadata_dim}\n")

Metadata dimension: 34



## 2. Hybrid Dataset and DataLoaders

In [12]:
class HybridDataset(Dataset):
    def __init__(self, hdf5_path, metadata_df, transform=None, is_test=False):
        self.hdf5_path = hdf5_path
        self.transform = transform
        self.is_test = is_test
        self.hdf5_file = None
        
        with h5py.File(hdf5_path, 'r') as f:
            available_ids = set(f.keys())
        
        self.metadata = metadata_df[
            metadata_df['isic_id'].isin(available_ids)
        ].reset_index(drop=True)
        
        feature_cols = [col for col in self.metadata.columns 
                       if col not in ['isic_id', 'target']]
        self.metadata_features = self.metadata[feature_cols].values.astype(np.float32)
        
        print(f"✓ {len(self.metadata)} samples from {Path(hdf5_path).name}")
        
        if not is_test and 'target' in self.metadata.columns:
            print(f"  Distribution: {self.metadata['target'].value_counts().to_dict()}")
    
    def _ensure_hdf5_open(self):
        if self.hdf5_file is None:
            self.hdf5_file = h5py.File(self.hdf5_path, 'r', swmr=True)
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        self._ensure_hdf5_open()
        
        row = self.metadata.iloc[idx]
        image_id = row['isic_id']
        
        img_array = self.hdf5_file[image_id][:]
        image = Image.fromarray(img_array)
        
        if self.transform:
            image = self.transform(image)
        
        metadata = torch.tensor(self.metadata_features[idx], dtype=torch.float32)
        
        if self.is_test:
            return image, metadata, image_id
        else:
            label = row['target']
            return image, metadata, label

In [13]:
# Transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split
train_df, val_df = train_test_split(
    train_meta_processed, test_size=0.2, random_state=42,
    stratify=train_meta_processed['target']
)

print(f"Split: {len(train_df):,} train / {len(val_df):,} val\n")

# Create datasets
train_dataset = HybridDataset(
    hdf5_path=data_dir / 'train-image-preprocessed.hdf5',
    metadata_df=train_df, transform=train_transform, is_test=False
)

val_dataset = HybridDataset(
    hdf5_path=data_dir / 'train-image-preprocessed.hdf5',
    metadata_df=val_df, transform=val_transform, is_test=False
)

test_dataset = HybridDataset(
    hdf5_path=data_dir / 'test-image-preprocessed.hdf5',
    metadata_df=test_meta_processed, transform=val_transform, is_test=True
)

# DataLoaders - SMALLER batch size for larger model!
BATCH_SIZE = 128  # Reduced from 256 (larger model uses more memory)
NUM_WORKERS = 16

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

print(f"DataLoaders: {BATCH_SIZE} batch (reduced for larger model), {NUM_WORKERS} workers")
print(f"Batches: {len(train_loader)} train, {len(val_loader)} val\n")

Split: 320,767 train / 80,192 val

✓ 320767 samples from train-image-preprocessed.hdf5
  Distribution: {0: 320493, 1: 274}
✓ 80192 samples from train-image-preprocessed.hdf5
  Distribution: {0: 80123, 1: 69}
✓ 100 samples from test-image-preprocessed.hdf5
DataLoaders: 128 batch (reduced for larger model), 16 workers
Batches: 2506 train, 627 val



## 3. Focal Loss

In [14]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = torch.exp(-bce_loss)
        focal_term = (1 - p_t) ** self.gamma
        focal_loss = self.alpha * focal_term * bce_loss
        return focal_loss.mean()

## 4. EfficientNetV2-M Hybrid Model

In [15]:
class MetadataProcessor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
    
    def forward(self, x):
        return self.fc(x)


class EfficientNetV2MHybrid(nn.Module):
    """
    Hybrid model with EfficientNetV2-M backbone (LARGER VERSION)
    
    Architecture:
      - EfficientNetV2-M (pretrained) → 1280 features
      - Metadata MLP → 64 features
      - Concatenate → 1344 features
      - Classifier → 1 output
    
    Upgrades from V2-S:
    - Parameters: 21.5M → 54M (2.5× larger)
    - ImageNet accuracy: 84.3% → 85.1% (+0.8%)
    - More capacity for learning complex patterns
    - Better pretrained representations
    """
    def __init__(self, metadata_dim):
        super().__init__()
        
        # Load pretrained EfficientNetV2-M (MEDIUM - larger than Small)
        self.efficientnet = efficientnet_v2_m(weights='IMAGENET1K_V1')
        
        # Remove original classifier
        self.efficientnet.classifier = nn.Identity()
        
        # Freeze early layers (80%)
        total_params = len(list(self.efficientnet.parameters()))
        freeze_until = int(total_params * 0.8)
        
        for idx, param in enumerate(self.efficientnet.parameters()):
            if idx < freeze_until:
                param.requires_grad = False
        
        # Metadata processor (same as before)
        self.metadata_processor = MetadataProcessor(metadata_dim)
        
        # Combined classifier
        # EfficientNetV2-M also outputs 1280 features (same as S!)
        self.classifier = nn.Sequential(
            nn.Linear(1280 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    
    def forward(self, image, metadata):
        img_features = self.efficientnet(image)
        meta_features = self.metadata_processor(metadata)
        combined = torch.cat([img_features, meta_features], dim=1)
        return self.classifier(combined)


# Create model
model = EfficientNetV2MHybrid(metadata_dim=metadata_dim).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Training Setup

In [16]:
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=2e-5)  # Lower LR, stronger regularization

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5
)

print("Training setup:")
print(f"  Loss: Focal Loss (alpha=0.25, gamma=2.0)")
print(f"  Optimizer: Adam (lr=0.0003, weight_decay=2e-5)")
print(f"  Note: Lower LR and stronger regularization vs V2-S")
print(f"  Reason: Larger model needs more careful training")
print(f"  Scheduler: ReduceLROnPlateau (patience=5)\n")

Training setup:
  Loss: Focal Loss (alpha=0.25, gamma=2.0)
  Optimizer: Adam (lr=0.0003, weight_decay=2e-5)
  Note: Lower LR and stronger regularization vs V2-S
  Reason: Larger model needs more careful training
  Scheduler: ReduceLROnPlateau (patience=5)



## 6. Training Functions

In [17]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    start_time = time.time()
    
    for images, metadata, labels in tqdm(loader, desc="Training", ncols=100):
        images = images.to(device, non_blocking=True)
        metadata = metadata.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping for larger model stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    elapsed = time.time() - start_time
    epoch_loss = running_loss / len(loader)
    epoch_auc = roc_auc_score(all_labels, all_preds)
    
    return epoch_loss, epoch_auc, elapsed


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, metadata, labels in tqdm(loader, desc="Validation", ncols=100):
            images = images.to(device, non_blocking=True)
            metadata = metadata.to(device, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(device, non_blocking=True)
            
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    epoch_auc = roc_auc_score(all_labels, all_preds)
    
    return epoch_loss, epoch_auc, all_preds, all_labels

## 7. Training Loop

In [ ]:
NUM_EPOCHS = 30  # More epochs for larger model
best_auc = 0.0
history = {
    'train_loss': [], 'train_auc': [], 'train_time': [],
    'val_loss': [], 'val_auc': []
}

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = Path('results') / f'efficientnetv2_m_hybrid_{timestamp}'
results_dir.mkdir(parents=True, exist_ok=True)

print("="*70)
print("STARTING TRAINING - EFFICIENTNETV2-M HYBRID")
print("="*70)
print(f"Results: {results_dir}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Model size: 2.5× larger than V2-S")
print(f"Expected improvement: 0.937 → 0.942-0.952 (+0.005-0.015 AUC)")
print("="*70 + "\n")

total_start = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")
    
    train_loss, train_auc, train_time = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    val_loss, val_auc, val_preds, val_labels = validate(
        model, val_loader, criterion, device
    )
    
    history['train_loss'].append(train_loss)
    history['train_auc'].append(train_auc)
    history['train_time'].append(train_time)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f} | Time: {train_time:.1f}s")
    print(f"  Val Loss:   {val_loss:.4f} | Val AUC:   {val_auc:.4f}")
    
    # Check for overfitting
    if epoch > 5 and train_auc > val_auc + 0.05:
        print(f"  ⚠ Warning: Train-Val gap = {train_auc - val_auc:.4f} (possible overfitting)")
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
        }, results_dir / 'best_efficientnetv2_m_hybrid.pth')
        print(f"  ✓ Saved best model (AUC: {best_auc:.4f})")
    
    scheduler.step(val_auc)
    print(f"  Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Memory monitoring
    if torch.cuda.is_available():
        memory_allocated = torch.cuda.max_memory_allocated() / 1e9
        print(f"  GPU memory used: {memory_allocated:.1f} GB")
    
    if optimizer.param_groups[0]['lr'] < 1e-6:
        print(f"\n  LR too small, stopping...")
        break

total_time = time.time() - total_start

# Save results
with open(results_dir / 'training_results.pkl', 'wb') as f:
    pickle.dump({
        'timestamp': timestamp,
        'model': 'EfficientNetV2-M Hybrid',
        'best_auc': best_auc,
        'history': history,
        'total_time': total_time,
        'batch_size': BATCH_SIZE,
        'metadata_dim': metadata_dim,
        'total_params': total_params,
        'trainable_params': trainable_params,
    }, f)

with open(results_dir / 'preprocessors.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'encoders': encoders}, f)

print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Best validation AUC: {best_auc:.4f}")
print(f"Improvement vs V2-S: {best_auc - 0.9508:.4f}")
print(f"Improvement vs baseline: {best_auc - 0.9365:.4f}")
print(f"\n✓ Results saved to: {results_dir}")
print(f"{'='*70}\n")

## 8. Evaluation

In [ ]:
checkpoint = torch.load(results_dir / 'best_efficientnetv2_m_hybrid.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
_, _, val_preds, val_labels = validate(model, val_loader, criterion, device)

# ROC curve
fpr, tpr, thresholds = roc_curve(val_labels, val_preds)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {best_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
plt.plot(fpr[optimal_idx], tpr[optimal_idx], 'go', markersize=12, 
         label=f'Threshold = {optimal_threshold:.4f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - EfficientNetV2-M Hybrid')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(results_dir / 'roc_curve.png', dpi=150)
print(f"✓ ROC curve saved\n")
plt.show()

# Training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (V2-M)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_auc'], label='Train', marker='o')
axes[1].plot(history['val_auc'], label='Val', marker='o')
axes[1].axhline(y=best_auc, color='r', linestyle='--', label=f'Best: {best_auc:.4f}')
axes[1].axhline(y=0.9508, color='gray', linestyle=':', label='V2-S: 0.9508')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('AUC (V2-M)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'training_history.png', dpi=150)
print(f"✓ Training plots saved\n")
plt.show()

## 9. Generate Predictions

In [ ]:
print("Generating test predictions...")
model.eval()
test_ids = []
test_preds = []

with torch.no_grad():
    for images, metadata, img_ids in tqdm(test_loader, desc="Testing", ncols=100):
        images = images.to(device, non_blocking=True)
        metadata = metadata.to(device, non_blocking=True)
        outputs = model(images, metadata)
        probs = torch.sigmoid(outputs).cpu().numpy()
        test_ids.extend(img_ids)
        test_preds.extend(probs.flatten())

submission = pd.DataFrame({'isic_id': test_ids, 'target': test_preds})
submission.to_csv(results_dir / 'submission_efficientnetv2_m_hybrid.csv', index=False)

print(f"\n{'='*70}")
print("SUBMISSION GENERATED")
print(f"{'='*70}")
print(f"File: submission_efficientnetv2_m_hybrid.csv")
print(f"Shape: {submission.shape}")
print(f"\nPrediction statistics:")
print(submission['target'].describe())
print(f"{'='*70}\n")

In [ ]:
# Evaluate the loaded model on validation set
print("\n" + "="*70)
print("EVALUATING LOADED MODEL ON VALIDATION SET")
print("="*70 + "\n")

model.eval()
val_loss, val_auc, val_preds, val_labels = validate(model, val_loader, criterion, device)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation AUC:  {val_auc:.4f}")
print(f"Checkpoint AUC:  {checkpoint['val_auc']:.4f}")
print(f"Difference:       {abs(val_auc - checkpoint['val_auc']):.4f}")

# ROC curve
fpr, tpr, thresholds = roc_curve(val_labels, val_preds)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {val_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
plt.plot(fpr[optimal_idx], tpr[optimal_idx], 'go', markersize=12, 
         label=f'Threshold = {optimal_threshold:.4f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Loaded EfficientNetV2-M Fixed Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Confusion matrix at optimal threshold
from sklearn.metrics import confusion_matrix, classification_report
val_pred_labels = (np.array(val_preds) > optimal_threshold).astype(int)
cm = confusion_matrix(val_labels, val_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.title(f'Confusion Matrix (threshold={optimal_threshold:.4f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(val_labels, val_pred_labels, 
                          target_names=['Benign', 'Malignant'],
                          digits=4))

In [ ]:
print("="*70)
print("OVERFITTING ANALYSIS")
print("="*70 + "\n")

final_train_auc = history['train_auc'][-1]
final_val_auc = history['val_auc'][-1]
best_val_auc = best_auc

train_val_gap = final_train_auc - final_val_auc

print(f"Final epoch:")
print(f"  Train AUC: {final_train_auc:.4f}")
print(f"  Val AUC:   {final_val_auc:.4f}")
print(f"  Gap:       {train_val_gap:.4f}")

print(f"\nBest model:")
print(f"  Val AUC: {best_val_auc:.4f}")

if train_val_gap > 0.08:
    print(f"\n  ⚠️ Large train-val gap! Model is overfitting.")
    print(f"  Recommendations:")
    print(f"    - Use V2-S instead (smaller, less prone to overfit)")
    print(f"    - Increase dropout")
    print(f"    - Add stronger augmentation")
elif train_val_gap > 0.05:
    print(f"\n  ⚠ Moderate overfitting. Acceptable but could be better.")
else:
    print(f"\n  ✓ Good generalization! Train-val gap is reasonable.")

print("\n" + "="*70)

In [ ]:
print("="*70)
print("OVERFITTING ANALYSIS")
print("="*70 + "\n")

final_train_auc = history['train_auc'][-1]
final_val_auc = history['val_auc'][-1]
best_val_auc = best_auc

train_val_gap = final_train_auc - final_val_auc

print(f"Final epoch:")
print(f"  Train AUC: {final_train_auc:.4f}")
print(f"  Val AUC:   {final_val_auc:.4f}")
print(f"  Gap:       {train_val_gap:.4f}")

print(f"\nBest model:")
print(f"  Val AUC: {best_val_auc:.4f}")

if train_val_gap > 0.08:
    print(f"\n  ⚠️ Large train-val gap! Model is overfitting.")
    print(f"  Recommendations:")
    print(f"    - Use V2-S instead (smaller, less prone to overfit)")
    print(f"    - Increase dropout")
    print(f"    - Add stronger augmentation")
elif train_val_gap > 0.05:
    print(f"\n  ⚠ Moderate overfitting. Acceptable but could be better.")
else:
    print(f"\n  ✓ Good generalization! Train-val gap is reasonable.")

print("\n" + "="*70)